# 05 - Simulation Demo
Replays scripted attack scenarios against the ThreatDetector and shows live SOC alerts.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from src.detection.detector import ThreatDetector
from src.detection.alert_engine import AlertEngine
print('Imports OK')

Imports OK

In [2]:
detector = ThreatDetector(models_dir='../models')
engine = AlertEngine(alerts_dir='../outputs/alerts')

# Doc's sample request
doc_result = detector.predict_doc_features(packet_size=1500, failed_logins=5, request_frequency=250)
print('Doc Sample (packet_size=1500, failed_logins=5, request_frequency=250):')
print(doc_result)

[2026-10-16 16:48:30] [INFO] [threat_detection]: ThreatDetector initialized with available model artifacts.


Doc Sample (packet_size=1500, failed_logins=5, request_frequency=250):

{'Threat_Detected': True, 'attack_type': 'brute_force', 'confidence': 1.0, 'severity': 'HIGH', 'input_features': {'packet_size': 1500, 'failed_logins': 5, 'request_frequency': 250}}

In [3]:
# DoS scenario
dos_record = {
    'src_ip': '185.220.101.5', 'dst_ip': '192.168.1.10',
    'protocol_type': 'tcp', 'service': 'http', 'flag': 'S0',
    'duration': 0.02, 'src_bytes': 15000, 'dst_bytes': 0,
    'packet_size': 1450, 'failed_logins': 0, 'request_frequency': 1400,
    'num_connections': 350, 'login_attempts': 0, 'unusual_hour': 0, 'geo_risk': 0.92
}
dos_result = detector.predict_network_record(dos_record)
print('DoS Scenario Result:', dos_result)
engine.emit_alert(dos_result, record=dos_record, print_console=True)

DoS Scenario Result:

{'Threat_Detected': True, 'attack_type': 'dos', 'confidence': 0.9083, 'severity': 'CRITICAL', 'anomaly_detected': False, 'anomaly_score': -0.1549, 'src_ip': '185.220.101.5', 'dst_ip': '192.168.1.10'}

[ALERT] [2026-10-16 16:48:30] [CRITICAL] DOS detected from 185.220.101.5 -> 192.168.1.10 (Conf: 0.91)

        Action: Rate-limit traffic and block source IP at edge firewall (Null-route / BGP Flowspec)

{'timestamp': '2026-10-16 16:48:30',
 'severity': 'CRITICAL',
 'attack_type': 'DOS',
 'confidence': 0.9083,
 'src_ip': '185.220.101.5',
 'dst_ip': '192.168.1.10',
 'recommended_action': 'Rate-limit traffic and block source IP at edge firewall (Null-route / BGP Flowspec)',
 'anomaly_score': -0.1549,
 'threat_detected': True}

In [4]:
# Phishing detector demo
phish_features = {
    'url_length': 115, 'has_ip_in_url': 1, 'num_special_chars': 14,
    'https_flag': 0, 'domain_age_days': 12, 'num_redirects': 3, 'suspicious_keywords': 2
}
phish_result = detector.predict_phishing(phish_features)
print('Phishing URL Analysis:', phish_result)

# Malware detector demo
malware_features = {
    'api_call_count': 920, 'registry_changes': 22, 'files_modified': 58,
    'network_connections': 18, 'entropy': 7.4, 'memory_usage_mb': 280.0, 'execution_time_sec': 8.5
}
malware_result = detector.predict_malware(malware_features)
print('Malware Host Analysis:', malware_result)

Phishing URL Analysis:

{'Threat_Detected': True, 'attack_type': 'phishing', 'confidence': 1.0, 'severity': 'HIGH'}

Malware Host Analysis:

{'Threat_Detected': True, 'attack_type': 'malware', 'confidence': 1.0, 'severity': 'CRITICAL'}

In [5]:
# Show stored alerts
recent = engine.get_recent_alerts(limit=10)
print(f'\nTotal stored alerts in log: {len(recent)}')
for a in recent[-3:]:
    print(f"  [{a['severity']}] {a['attack_type']} from {a['src_ip']} at {a['timestamp']}")


Total stored alerts in log: 10

  [HIGH] BRUTE_FORCE from 127.0.0.1 at 2026-10-16 16:42:59

  [HIGH] BRUTE_FORCE from 127.0.0.1 at 2026-10-16 16:44:06

  [CRITICAL] DOS from 185.220.101.5 at 2026-10-16 16:48:30